In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline


url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
path = Path("names.txt")

if not path.exists():
    urlretrieve(url, path)

words = path.read_text().splitlines()

print(len(words))
print(words[:10])

32033
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [4]:
ch = sorted(list(set(''.join(words))))

In [17]:
sTOi = {s:i+1 for i,s in enumerate(ch)}

In [18]:
sTOi['.'] = 0

In [19]:
sTOi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [20]:
B = torch.zeros((27,27), dtype=torch.int32)

In [22]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = sTOi[ch1]
        ix2 = sTOi[ch2]
        B[ix1,ix2] +=1

In [21]:
iTOs = {i:s for s,i in sTOi.items()}

In [39]:
P = (B+1).float()
P /= P.sum(1, keepdim=True)

In [40]:
for i in range(5):

    out = []
    ix=0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True).item()
        out.append(iTOs[ix])

        if ix==0:
            break
    print(''.join(out))

a.
iri.
san.
ca.
dan.


In [32]:
#interesting how it got some real words. kinda cool for a bigram

In [44]:
llh = 0.0
n=0
for w in ["asad"]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = sTOi[ch1]
        ix2 = sTOi[ch2]
        prb = P[ix1, ix2]
        logprb = torch.log(prb)
        llh += logprb
        n += 1

print(f'{-llh/n}')

2.6314170360565186


In [45]:
#looks like my name is not too out of the norm

In [85]:
xs, ys = [], []


for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = sTOi[ch1]
        ix2 = sTOi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

In [86]:
W = torch.randn((27,27), requires_grad=True)

In [87]:
for k in range(100):
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())

    W.grad = None
    loss.backward()

    W.data += -75 * W.grad

3.7415337562561035
3.248185873031616
2.991874933242798
2.8581655025482178
2.7772467136383057
2.7222187519073486
2.683014154434204
2.652675151824951
2.629443883895874
2.6103644371032715
2.595120668411255
2.5820229053497314
2.5712711811065674
2.561755895614624
2.5538394451141357
2.5466835498809814
2.54068660736084
2.535160779953003
2.530506134033203
2.526134967803955
2.522439956665039
2.518903970718384
2.5159096717834473
2.5129916667938232
2.5105206966400146
2.508070230484009
2.505998373031616
2.5039069652557373
2.5021445751190186
2.500335216522217
2.49881649017334
2.497231960296631
2.495908498764038
2.4945056438446045
2.4933414459228516
2.4920876026153564
2.4910547733306885
2.4899251461029053
2.489001989364624
2.4879772663116455
2.4871480464935303
2.486212730407715
2.485463857650757
2.484605550765991
2.4839258193969727
2.483135461807251
2.4825167655944824
2.481785297393799
2.481220245361328
2.480541467666626
2.4800243377685547
2.4793925285339355
2.4789178371429443
2.478327751159668
2.47

In [83]:
num

228146

In [88]:
#interesting loss rate. nn performs similar to the counting bigram, same model fundamentally